# 権限ではなく実績を監査する——Sensitive Data Access レポートで機密データへの実アクセスを追跡 — 検証ノートブック

## このノートブックについて

Zenn 記事「[権限ではなく実績を監査する——Sensitive Data Access レポートで機密データへの実アクセスを追跡](https://zenn.dev/gtk0326/articles/i73-feature-update-2026-05-28-sensitive-data-acc)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## ステップ1: 分類結果を確認する


In [ ]:
SELECT column_name, tag_name, tag_value
FROM TABLE(
    SNOWFLAKE.INFORMATION_SCHEMA.TAG_REFERENCES_WITH_LINEAGE(
        'demo_governance_db.pii_data.customers',
        'TABLE'
    )
)
ORDER BY column_name;

## ステップ2: SNOWFLAKE.DATA_SECURITY スキーマのビュー構造を確認する

Access レポートが生成するビューは `SNOWFLAKE.DATA_SECURITY` スキーマに作成されます。同スキーマの `ENTITLEMENT_REPORT` ビューの列構造を確認しておきます。


In [ ]:
SELECT column_name, data_type
FROM SNOWFLAKE.INFORMATION_SCHEMA.COLUMNS
WHERE table_catalog = 'SNOWFLAKE'
  AND table_schema  = 'DATA_SECURITY'
  AND table_name    = 'ENTITLEMENT_REPORT'
ORDER BY ordinal_position;

## ステップ3: Sensitive Data Access レポートを生成する


In [ ]:
CALL SNOWFLAKE.DATA_SECURITY.GENERATE_SENSITIVE_DATA_ACCESS_REPORT(
    LOOKBACK_DAYS => 30
);

## ステップ4: 生成されたビューからアクセス実績を取得する


In [ ]:
SELECT
    user_name,
    table_database,
    table_schema,
    table_name,
    role_name
FROM SNOWFLAKE.DATA_SECURITY.SENSITIVE_DATA_ACCESS_REPORT
ORDER BY user_name, table_database, table_schema, table_name;

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** Dynamic Table などを残すとバックグラウンドでリフレッシュが継続しクレジットが消費されます。